In [2]:
!pip install -U huggingface_hub==0.30.1
!pip install -U transformers==4.41.2
!pip install -U sentence-transformers==2.6.1


  Using cached huggingface_hub-0.30.1-py3-none-any.whl.metadata (13 kB)
Using cached huggingface_hub-0.30.1-py3-none-any.whl (481 kB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.14.1
    Uninstalling huggingface-hub-0.14.1:
      Successfully uninstalled huggingface-hub-0.14.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2025.7.0 which is incompatible.


In [3]:
!pip uninstall transformers -y
!pip uninstall sentence-transformers -y
!pip uninstall huggingface_hub -y

Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2
Found existing installation: sentence-transformers 2.6.1
Uninstalling sentence-transformers-2.6.1:
  Successfully uninstalled sentence-transformers-2.6.1
Found existing installation: huggingface-hub 0.30.1
Uninstalling huggingface-hub-0.30.1:
  Successfully uninstalled huggingface-hub-0.30.1


In [4]:
!pip install huggingface_hub==0.30.1
!pip install transformers==4.41.2
!pip install sentence-transformers==2.6.1

  Using cached huggingface_hub-0.30.1-py3-none-any.whl.metadata (13 kB)
Using cached huggingface_hub-0.30.1-py3-none-any.whl (481 kB)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2025.7.0 which is incompatible.


  Using cached transformers-4.41.2-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.41.2-py3-none-any.whl (9.1 MB)
  Using cached sentence_transformers-2.6.1-py3-none-any.whl.metadata (11 kB)
Using cached sentence_transformers-2.6.1-py3-none-any.whl (163 kB)


In [5]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Mô hình đã sẵn sàng.")

ImportError: cannot import name 'WeakFileLock' from 'huggingface_hub.utils' (c:\Users\DELL\anaconda3\Lib\site-packages\huggingface_hub\utils\__init__.py)

In [13]:
import os
import uuid
import json
import gradio as gr
import chromadb
from sentence_transformers import SentenceTransformer
from langchain.llms import Ollama

# SETUP
client = chromadb.CloudClient(
    api_key='ck-8p54Pf1QCBubbfd5UDRnRb7iMgC3U6fENhBUgwMgNuFA',
    tenant='10979a1e-b119-4b7d-9495-dec65ed83dc1',
    database='datachatbox'
)

collection = client.get_or_create_collection(name="memory")

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
llm = Ollama(model="qwen3:4b")  # dùng mô hình cục bộ Qwen3:4b

def generate_user_id():
    return str(uuid.uuid4())

def create_summary(chat_history):
    if not chat_history:
        return "Không có nội dung"
    u, b = chat_history[-1]
    return f"{u[:20]} → {b[:20]}"

def chat(user_input, history, user_id):
    # Lấy trí nhớ gần nhất (nếu có)
    memory_context = semantic_search(user_input)
    if memory_context:
        context_text = "\n".join([f"Người: {u}\nBot: {b}" for u, b in memory_context])
        prompt = f"{context_text}\n\nNgười: {user_input}\nBot:"
    else:
        prompt = user_input

    # Gửi prompt đến mô hình
    response = llm(prompt)
    history.append((user_input, response))
    return history, history

def save_memory_to_chroma(user_id, summary, full_text):
    embedding = embed_model.encode(summary).tolist()
    collection.add(
        documents=[summary],
        embeddings=[embedding],
        metadatas=[{
            "user_id": user_id,
            "full_text": json.dumps(full_text, ensure_ascii=False)  # CHUYỂN LIST → JSON STRING
        }],
        ids=[user_id + "_" + str(len(collection.get()['ids']))]
    )

def save_hybrid_memory(user_id, chat_history):
    if not chat_history:
        return [], gr.update(choices=[]), "Không có nội dung để lưu."

    summary = create_summary(chat_history)
    full_text = chat_history.copy()
    save_memory_to_chroma(user_id, summary, full_text)

    updated_choices = summary_list()
    return [], gr.update(choices=updated_choices), f"Đã lưu: {summary}"

def summary_list():
    result = collection.get()
    return result["documents"] if "documents" in result else []

def load_conversation(selected_summary):
    result = collection.get()
    for doc, meta in zip(result["documents"], result["metadatas"]):
        if doc == selected_summary:
            return json.loads(meta["full_text"])  # CHUYỂN JSON STRING → LIST
    return []

def semantic_search(query_text):
    query_embedding = embed_model.encode(query_text).tolist()
    results = collection.query(query_embeddings=[query_embedding], n_results=1)
    if results and results["metadatas"][0]:
        return json.loads(results["metadatas"][0][0]["full_text"])  # CHUYỂN JSON STRING → LIST
    return []

# GIAO DIỆN
with gr.Blocks() as demo:
    gr.Markdown("Hybrid Chatbot")

    with gr.Row():
        with gr.Column(scale=1):
            saved_chats = gr.Radio(label="Đoạn chat đã lưu", choices=[], interactive=True)
            search_box = gr.Textbox(placeholder="Đoạn chat gần nhất...")
            search_btn = gr.Button("Gợi nhớ")

        with gr.Column(scale=4):
            chatbot = gr.Chatbot()
            msg = gr.Textbox(placeholder="Nhập câu hỏi...")
            user_id = gr.Textbox(value=generate_user_id(), visible=False)
            state = gr.State([])

            with gr.Row():
                send = gr.Button("Gửi")
                save_btn = gr.Button("Lưu & Tạo mới")
                status = gr.Textbox(label="Trạng thái", interactive=False)

    send.click(chat, [msg, state, user_id], [chatbot, state])
    save_btn.click(save_hybrid_memory, [user_id, state], [state, saved_chats, status])
    saved_chats.change(load_conversation, saved_chats, chatbot)
    search_btn.click(semantic_search, [search_box], chatbot)

demo.launch()


c:\Users\DELL\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  local_dir=local_dir,
C:\Users\DELL\AppData\Local\Temp\ipykernel_17748\3099007746.py:96: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\DELL\anaconda3\Lib\site-packages\gradio\queueing.py", line 625, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\DELL\anaconda3\Lib\site-packages\gradio\route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\DELL\anaconda3\Lib\site-packages\gradio\blocks.py", line 2146, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\DELL\anaconda3\Lib\site-packages\gradio\blocks.py", line 1664, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\DELL\anaconda3\Lib\site-packages\anyio\to_thread.py", line 56, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
      

In [3]:
import os
import uuid
import json
import gradio as gr
import chromadb
from sentence_transformers import SentenceTransformer
from langchain.llms import Ollama

# SETUP
client = chromadb.CloudClient(
    api_key='ck-8p54Pf1QCBubbfd5UDRnRb7iMgC3U6fENhBUgwMgNuFA',
    tenant='10979a1e-b119-4b7d-9495-dec65ed83dc1',
    database='datachatbox'
)

collection = client.get_or_create_collection(name="memory")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
llm = Ollama(model="qwen3:4b")

def generate_user_id():
    return str(uuid.uuid4())

def create_summary(chat_history):
    if not chat_history:
        return "Không có nội dung"
    u, b = chat_history[-1]
    return f"{u[:20]} → {b[:20]}"

def safe_load_json(data):
    if isinstance(data, str):
        try:
            return json.loads(data)
        except Exception:
            return []
    return data

def chat(user_input, history, user_id):
    memory_context = semantic_search(user_input)

    if isinstance(memory_context, list) and all(isinstance(item, (list, tuple)) and len(item) == 2 for item in memory_context):
        context_text = "\n".join([f"Người: {u}\nBot: {b}" for u, b in memory_context])
        prompt = f"{context_text}\n\nNgười: {user_input}\nBot:"
    else:
        prompt = user_input

    response = llm(prompt)
    history.append((user_input, response))
    return history, history

def save_memory_to_chroma(user_id, summary, full_text):
    embedding = embed_model.encode(summary).tolist()
    collection.add(
        documents=[json.dumps(full_text, ensure_ascii=False)],
        embeddings=[embedding],
        metadatas=[{
            "user_id": user_id,
            "summary": summary
        }],
        ids=[user_id + "_" + str(len(collection.get()['ids']))]
    )

def save_hybrid_memory(user_id, chat_history):
    if not chat_history:
        return [], gr.update(choices=[]), "Không có nội dung để lưu."

    summary = create_summary(chat_history)
    full_text = chat_history.copy()
    save_memory_to_chroma(user_id, summary, full_text)

    updated_choices = summary_list()
    return [], gr.update(choices=updated_choices), f"Đã lưu: {summary}"

def summary_list():
    result = collection.get()
    return [meta["summary"] for meta in result["metadatas"] if "summary" in meta]

def load_conversation(selected_summary):
    result = collection.get()
    for doc, meta in zip(result["documents"], result["metadatas"]):
        if meta.get("summary") == selected_summary:
            return safe_load_json(doc)
    return []

def semantic_search(query_text):
    query_embedding = embed_model.encode(query_text).tolist()
    results = collection.query(query_embeddings=[query_embedding], n_results=3)  # tăng n_results lên 3
    if results and results["documents"][0]:
        doc = safe_load_json(results["documents"][0])
        if isinstance(doc, list) and all(isinstance(item, (list, tuple)) and len(item) == 2 for item in doc):
            return doc
    return []

# GIAO DIỆN
with gr.Blocks() as demo:
    gr.Markdown("Hybrid Chatbot")

    with gr.Row():
        with gr.Column(scale=1):
            saved_chats = gr.Radio(label="Đoạn chat đã lưu", choices=[], interactive=True)
            search_box = gr.Textbox(placeholder="Nhập từ khóa gợi nhớ...")
            search_btn = gr.Button("Gợi nhớ")

        with gr.Column(scale=4):
            chatbot = gr.Chatbot()
            msg = gr.Textbox(placeholder="Nhập câu hỏi...")
            user_id = gr.Textbox(value=generate_user_id(), visible=False)
            state = gr.State([])

            with gr.Row():
                send = gr.Button("Gửi")
                save_btn = gr.Button("Lưu & Tạo mới")
                clear_btn = gr.Button("Tạo trang mới")  # nút mới
                status = gr.Textbox(label="Trạng thái", interactive=False)

    send.click(chat, [msg, state, user_id], [chatbot, state])
    save_btn.click(save_hybrid_memory, [user_id, state], [state, saved_chats, status])
    saved_chats.change(load_conversation, saved_chats, chatbot)
    search_btn.click(semantic_search, [search_box], chatbot)

    # Xử lý nút tạo trang mới (reset khung chat + state)
    clear_btn.click(fn=lambda: ([], []), outputs=[chatbot, state])

demo.launch()


c:\Users\DELL\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\DELL\AppData\Local\Temp\ipykernel_19844\3283201371.py:104: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [6]:
!pip install --upgrade gradio huggingface_hub


  Using cached huggingface_hub-0.33.5-py3-none-any.whl.metadata (14 kB)
   ---------------------------------------- 0.0/59.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/59.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/59.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/59.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/59.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/59.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/59.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/59.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/59.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/59.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/59.5 MB ? eta -:--:--
    --------------------------------------- 0.8/59.5 MB 2.4 MB/s eta 0:00:25
    --------------------------------------- 1.3/59.5 MB 2.2 MB/s eta 0:00:26
   -

ERROR: Exception:
Traceback (most recent call last):
  File "C:\Users\DELL\anaconda3\Lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "C:\Users\DELL\anaconda3\Lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ^^^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\anaconda3\Lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ^^^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\anaconda3\Lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 98, in read
    data: bytes = self.__fp.read(amt)
                  ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\anaconda3\Lib\http\client.py", line 479, in read
    s = self.fp.read(amt)
        ^^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\anaconda3\Lib\socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^